# CrisisOps v2 — Kaggle T4x2 GRPO Notebook

This notebook is pre-configured for Kaggle **GPU T4 x2** with **Qwen/Qwen2.5-3B-Instruct** and GRPO.

## Required pre-run settings (Kaggle UI)
- Accelerator: **GPU T4 x2**
- Internet: **ON**
- Secrets: optionally add `HF_TOKEN` for Hub backups

## Key training settings
- `num_generations = 8`
- `per_device_train_batch_size = 4`
- `gradient_accumulation_steps = 2`
- checkpoint every **50** steps to `/kaggle/output/checkpoints`
- final model to `/kaggle/output/crisisops_final`

## Curriculum guardrails
- Start at Level 1
- Unlock threshold = 0.20
- Rolling window = 50 episodes

Run all cells top-to-bottom.

In [ ]:
# Cell 1: Kaggle install cell
%matplotlib inline

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q unsloth trl accelerate peft datasets huggingface_hub
!pip install -q openenv

print("✅ Kaggle dependencies installed")

In [ ]:
# Cell 2: Kaggle runtime preflight checks
import os
import torch
import requests

assert os.path.exists("/kaggle"), "This notebook is intended for Kaggle runtime."

os.makedirs("/kaggle/output/checkpoints", exist_ok=True)
print("✅ /kaggle/output/checkpoints writable")

if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f"GPU count: {n}")
    for i in range(n):
        print(f"  GPU[{i}]: {torch.cuda.get_device_name(i)}")
    assert n >= 2, "Expected Kaggle GPU T4 x2."
else:
    raise RuntimeError("GPU not detected.")

try:
    r = requests.get("https://huggingface.co", timeout=10)
    print("Internet check status:", r.status_code)
except Exception as e:
    raise RuntimeError(f"Internet appears disabled: {e}")

print("✅ Kaggle preflight basic checks passed")

In [ ]:
# Cell 3: Clone repository into Kaggle working directory
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/vedchamp07/crisisops"
REPO_PATH = "/kaggle/working/crisisops"

if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)

os.chdir("/kaggle/working")
!git clone {REPO_URL} {REPO_PATH}

assert os.path.exists(os.path.join(REPO_PATH, "env")), "Clone failed"
os.chdir(REPO_PATH)

Path("/kaggle/output/checkpoints").mkdir(parents=True, exist_ok=True)
Path("/kaggle/output/plots").mkdir(parents=True, exist_ok=True)

print("✅ Repo ready:", os.getcwd())

In [ ]:
# Cell 4: Secrets (optional HF hub backup)
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    print("✅ HF_TOKEN loaded from Kaggle Secrets")
except Exception:
    print("ℹ️ HF_TOKEN not configured; skipping Hub backup steps")

In [ ]:
# Cell 5: Accelerate multi-GPU init (critical for T4x2)
from accelerate import Accelerator
accelerator = Accelerator()
print(f"Using {accelerator.num_processes} GPU(s)")

In [ ]:
# Cell 6: Imports + Kaggle training config
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback, AutoTokenizer

from env.environment import CrisisOpsEnv
from env.crisis_generator import CrisisGenerator
from calibration.calibrate import run_calibration
from training.grpo_trainer import (
    SYSTEM_PROMPT,
    format_observation_as_prompt,
    parse_action_from_response,
    build_training_dataset,
    _make_reward_fn,
)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
LOAD_IN_4BIT = True
MAX_SEQ_LENGTH = 2048

# Requested parameters
NUM_GENERATIONS = 8
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
CHECKPOINT_EVERY_STEPS = 50
MAX_STEPS = 1200

# Curriculum settings
START_LEVEL = 1
UNLOCK_THRESHOLD = 0.20
UNLOCK_WINDOW = 50

OUTPUT_ROOT = "/kaggle/output"
OUTPUT_DIR = f"{OUTPUT_ROOT}/crisisops_grpo"
CHECKPOINT_DIR = f"{OUTPUT_ROOT}/checkpoints"
PLOTS_DIR = f"{OUTPUT_ROOT}/plots"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

print("✅ Config ready")
print("MODEL_NAME:", MODEL_NAME)
print("NUM_GENERATIONS:", NUM_GENERATIONS)
print("per_device_train_batch_size:", PER_DEVICE_TRAIN_BATCH_SIZE)
print("gradient_accumulation_steps:", GRADIENT_ACCUMULATION_STEPS)
print("effective batch:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * max(1, accelerator.num_processes))

In [ ]:
# Cell 7: Calibration gate (must pass before training)
cal = run_calibration(n_episodes=20, seed=42)

greedy_mean = cal["greedy_mean"]
oracle_mean = cal["oracle_mean"]
gap = cal["gap"]

print(f"Greedy: {greedy_mean:.3f} | Oracle: {oracle_mean:.3f} | Gap: {gap:.3f}")
assert 0.20 <= gap <= 0.35, f"Reward gap {gap:.3f} out of target range — fix env before training"

In [ ]:
# Cell 8: Observation sanity check
obs_env = CrisisOpsEnv()
obs = obs_env.reset(seed=42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
prompt = (
    "You are an AI project manager. Choose exactly one action name from the valid set.\n"
    f"Observation:\n{format_observation_as_prompt(obs)}\n"
    "Answer with only the action name."
)
inputs = tokenizer(prompt, return_tensors="pt")

print("Prompt token length:", inputs["input_ids"].shape[1])
print("First 500 chars of prompt:")
print(prompt[:500])
assert inputs["input_ids"].shape[1] < 1024, "Prompt too long; shorten observation/template"

In [ ]:
# Cell 9: Model load (Unsloth 4-bit) + dataset
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    device_map="auto",
)

model.generation_config.max_length = None

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

generator = CrisisGenerator(curriculum_level=START_LEVEL)
train_dataset = build_training_dataset(
    scenario_fn_or_generator=generator,
    curriculum_level=START_LEVEL,
    n_samples=max(1200, MAX_STEPS),
    seed=42,
)

print("✅ Model and dataset ready")
print("Dataset size:", len(train_dataset))

In [ ]:
# Cell 10: Reward shaping + callbacks + GRPOTrainer run
VALID_ACTIONS = {
    "query_status", "query_member_report",
    "query_observable_signals", "query_ticket",
    "reassign_task", "communicate", "cut_scope",
    "escalate_risk", "request_resource",
    "update_timeline", "consult_expert",
    "resolve_blocker", "submit_recovery_plan",
}
VERIF_ACTIONS = {"query_observable_signals", "query_member_report", "query_status", "query_ticket"}

metrics = {
    "reward": [],
    "cvr": [],
    "step": [],
    "level": [],
}


def compute_reward(action_str, counterfactual_reward):
    format_bonus = 0.1 if action_str.strip() in VALID_ACTIONS else -0.1
    return counterfactual_reward + format_bonus


base_reward_fn = _make_reward_fn(generator, START_LEVEL, model, tokenizer)


def reward_fn(prompts, completions, **kwargs):
    base_rewards = base_reward_fn(prompts, completions, **kwargs)
    final_rewards = []
    verif_count = 0

    for completion, cf_reward in zip(completions, base_rewards):
        parsed = parse_action_from_response(completion)
        action = parsed.get("action_type", "")
        if action in VERIF_ACTIONS:
            verif_count += 1
        final_rewards.append(float(compute_reward(action, cf_reward)))

    batch_cvr = verif_count / max(1, len(completions))
    metrics["reward"].append(float(np.mean(final_rewards)))
    metrics["cvr"].append(float(batch_cvr))
    metrics["step"].append(len(metrics["step"]) + 1)
    metrics["level"].append(generator.curriculum_level)

    return final_rewards


class KaggleProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        step = int(state.global_step)

        if metrics["reward"]:
            reward_val = metrics["reward"][-1]
            cvr_val = metrics["cvr"][-1]
            print(f"step={step} reward={reward_val:.4f} cvr={cvr_val:.4f} level={generator.curriculum_level}")

            if len(metrics["reward"]) >= UNLOCK_WINDOW:
                rolling = float(np.mean(metrics["reward"][-UNLOCK_WINDOW:]))
                if generator.curriculum_level == 1 and rolling > UNLOCK_THRESHOLD:
                    generator.curriculum_level = 2
                    print(f"Curriculum advanced to Level 2 at step {step}")

        if step > 0 and step % CHECKPOINT_EVERY_STEPS == 0:
            ckpt_path = f"{CHECKPOINT_DIR}/ckpt_step{step}"
            model.save_pretrained(ckpt_path)
            tokenizer.save_pretrained(ckpt_path)
            print(f"Saved checkpoint at step {step} -> {ckpt_path}")

            # Plot every 100 steps
            if step % 100 == 0 and metrics["step"]:
                fig, ax1 = plt.subplots(figsize=(10, 4))
                ax1.plot(metrics["step"], metrics["reward"], label="reward", color="tab:blue")
                ax1.set_xlabel("step")
                ax1.set_ylabel("reward", color="tab:blue")
                ax2 = ax1.twinx()
                ax2.plot(metrics["step"], metrics["cvr"], label="cvr", color="tab:green")
                ax2.set_ylabel("cvr", color="tab:green")
                plt.title("Training reward/CVR")
                plt.tight_layout()
                plot_path = f"{PLOTS_DIR}/curve_step{step}.png"
                plt.savefig(plot_path, dpi=140)
                plt.close(fig)
                print(f"Saved plot -> {plot_path}")

        return control


training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_steps=MAX_STEPS,
    logging_steps=1,
    save_steps=CHECKPOINT_EVERY_STEPS,
    learning_rate=2e-5,
    max_prompt_length=1024,
    max_completion_length=256,
    temperature=0.1,
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    reward_funcs=reward_fn,
    train_dataset=train_dataset,
    callbacks=[KaggleProgressCallback()],
)

trainer.train()

In [ ]:
# Cell 11: Final save + optional Hub backup
final_dir = "/kaggle/output/crisisops_final"
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"✅ Final model saved to {final_dir}")

# Optional backup to Hub if HF_TOKEN exists
if hf_token:
    try:
        from huggingface_hub import login
        login(token=hf_token)
        print("HF login OK. You can now push checkpoints manually if needed.")
    except Exception as e:
        print(f"HF login failed: {e}")

print("Done.")